# Making nutrition table

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
SELECT *
FROM products;

In [0]:
# name, brandname, url, imageUrl, pricePerUnit, calcPricePerUnit, Price, currentPrice




In [0]:
%sql
SELECT
    get_json_object(product_json, '$.title') AS product,

    CAST(get_json_object(product_json, '$.comparePricePerUnit') AS DOUBLE)
        AS price_per_kg,

    CAST(get_json_object(product_json, '$.nutritionalContent[6].amount') AS DOUBLE)
        AS protein_per_100g,

    ROUND(
        CAST(get_json_object(product_json, '$.nutritionalContent[6].amount') AS DOUBLE) * 10
        /
        CAST(get_json_object(product_json, '$.comparePricePerUnit') AS DOUBLE),
        2
    ) AS protein_per_nok

FROM products

WHERE get_json_object(product_json, '$.compareUnit') = 'kg'

ORDER BY protein_per_nok DESC

LIMIT 10;

# BRONZE

In [0]:
%sql
CREATE OR REPLACE TABLE product_nutritiens_bronze AS
SELECT
  chain,

  get_json_object(product_json, '$.title') AS title,
  get_json_object(product_json, '$.brand') AS brand,
  get_json_object(product_json, '$.slugifiedUrl') AS slugifiedUrl,
  get_json_object(product_json, '$.imagePath') AS imagePath,

  CAST(get_json_object(product_json, '$.pricePerUnit') AS STRING) AS pricePerUnit,
  CAST(get_json_object(product_json, '$.comparePricePerUnit') AS STRING) AS comparePricePerUnit,
  get_json_object(product_json, '$.compareUnit') AS compareUnit,

  get_json_object(product_json, '$.subtitle') AS subtitle,
  get_json_object(product_json, '$.description') AS description,
  get_json_object(product_json, '$.store.name') AS storeName,
  get_json_object(product_json, '$.ean') AS ean,

  CAST(get_json_object(product_json, '$.nutritionalContent[0].amount') AS STRING) AS energyAmount,
  get_json_object(product_json, '$.nutritionalContent[0].unit') AS energyUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[1].amount') AS STRING) AS caloriesAmount,
  get_json_object(product_json, '$.nutritionalContent[1].unit') AS caloriesUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[2].amount') AS STRING) AS fatAmount,
  get_json_object(product_json, '$.nutritionalContent[2].unit') AS fatUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[3].amount') AS STRING) AS saturatedFatAmount,
  get_json_object(product_json, '$.nutritionalContent[3].unit') AS saturatedFatUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[4].amount') AS STRING) AS carbohydratesAmount,
  get_json_object(product_json, '$.nutritionalContent[4].unit') AS carbohydratesUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[5].amount') AS STRING) AS sugarsAmount,
  get_json_object(product_json, '$.nutritionalContent[5].unit') AS sugarsUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[6].amount') AS STRING) AS proteinAmount,
  get_json_object(product_json, '$.nutritionalContent[6].unit') AS proteinUnit,

  CAST(get_json_object(product_json, '$.nutritionalContent[7].amount') AS STRING) AS saltAmount,
  get_json_object(product_json, '$.nutritionalContent[7].unit') AS saltUnit

FROM products
WHERE CAST(get_json_object(product_json, '$.comparePricePerUnit') AS STRING) > '0'
  AND get_json_object(product_json, '$.compareUnit') IN ('kg', 'l');

In [0]:
%sql
SELECT *
FROM product_nutritiens_bronze;

# SILVER

## INSPECTION

In [0]:
%sql
SELECT
  100.0 * COUNT_IF(fatUnit = 'g') / COUNT(*) AS percentage
FROM product_nutritiens_bronze;

In [0]:
%sql
SELECT DISTINCT fatUnit
FROM product_nutritiens_bronze;

In [0]:
%sql
SELECT
  100.0 * COUNT_IF(chain IS NULL) / COUNT(*) AS chain_null_percent,
  100.0 * COUNT_IF(title IS NULL) / COUNT(*) AS title_null_percent,
  100.0 * COUNT_IF(brand IS NULL) / COUNT(*) AS brand_null_percent,
  100.0 * COUNT_IF(slugifiedUrl IS NULL) / COUNT(*) AS slugifiedUrl_null_percent,
  100.0 * COUNT_IF(imagePath IS NULL) / COUNT(*) AS imagePath_null_percent,
  100.0 * COUNT_IF(pricePerUnit IS NULL) / COUNT(*) AS pricePerUnit_null_percent,
  100.0 * COUNT_IF(comparePricePerUnit IS NULL) / COUNT(*) AS comparePricePerUnit_null_percent,
  100.0 * COUNT_IF(compareUnit IS NULL) / COUNT(*) AS compareUnit_null_percent,
  100.0 * COUNT_IF(subtitle IS NULL) / COUNT(*) AS subtitle_null_percent,
  100.0 * COUNT_IF(description IS NULL) / COUNT(*) AS description_null_percent,
  100.0 * COUNT_IF(storeName IS NULL) / COUNT(*) AS storeName_null_percent,
  100.0 * COUNT_IF(ean IS NULL) / COUNT(*) AS ean_null_percent,
  100.0 * COUNT_IF(energyAmount IS NULL) / COUNT(*) AS energyAmount_null_percent,
  100.0 * COUNT_IF(energyUnit IS NULL) / COUNT(*) AS energyUnit_null_percent,
  100.0 * COUNT_IF(caloriesAmount IS NULL) / COUNT(*) AS caloriesAmount_null_percent,
  100.0 * COUNT_IF(caloriesUnit IS NULL) / COUNT(*) AS caloriesUnit_null_percent,
  100.0 * COUNT_IF(fatAmount IS NULL) / COUNT(*) AS fatAmount_null_percent,
  100.0 * COUNT_IF(fatUnit IS NULL) / COUNT(*) AS fatUnit_null_percent,
  100.0 * COUNT_IF(saturatedFatAmount IS NULL) / COUNT(*) AS saturatedFatAmount_null_percent,
  100.0 * COUNT_IF(saturatedFatUnit IS NULL) / COUNT(*) AS saturatedFatUnit_null_percent,
  100.0 * COUNT_IF(carbohydratesAmount IS NULL) / COUNT(*) AS carbohydratesAmount_null_percent,
  100.0 * COUNT_IF(carbohydratesUnit IS NULL) / COUNT(*) AS carbohydratesUnit_null_percent,
  100.0 * COUNT_IF(sugarsAmount IS NULL) / COUNT(*) AS sugarsAmount_null_percent,
  100.0 * COUNT_IF(sugarsUnit IS NULL) / COUNT(*) AS sugarsUnit_null_percent,
  100.0 * COUNT_IF(proteinAmount IS NULL) / COUNT(*) AS proteinAmount_null_percent,
  100.0 * COUNT_IF(proteinUnit IS NULL) / COUNT(*) AS proteinUnit_null_percent,
  100.0 * COUNT_IF(saltAmount IS NULL) / COUNT(*) AS saltAmount_null_percent,
  100.0 * COUNT_IF(saltUnit IS NULL) / COUNT(*) AS saltUnit_null_percent
FROM product_nutritiens_bronze;

In [0]:
%sql
SELECT * FROM product_nutritiens_bronze
WHERE fatUnit IS NULL

## MAKING TABLE
- Removing all -Unit columns, as they contain only a single non-null value.
- Removing storeName, as all values are null.
- Removing rows where -Amount is null, as they can't be ranked.
- All numerical values are cast to double.




In [0]:
%sql
CREATE OR REPLACE TABLE product_nutritiens_silver AS
SELECT
  chain,
  title,
  brand,
  slugifiedUrl AS website_url,
  imagePath AS image_path,

  CAST(pricePerUnit AS DOUBLE) AS price_per_unit,
  CAST(comparePricePerUnit AS DOUBLE) AS compare_price_per_unit,
  compareUnit AS compare_unit,

  subtitle,
  description,
  ean,

  CAST(energyAmount AS DOUBLE) AS energy_amount,
  CAST(caloriesAmount AS DOUBLE) AS calories_amount,
  CAST(fatAmount AS DOUBLE) AS fat_amount,
  CAST(saturatedFatAmount AS DOUBLE) AS saturated_fat_amount,
  CAST(carbohydratesAmount AS DOUBLE) AS carbohydrates_amount,
  CAST(sugarsAmount AS DOUBLE) AS sugars_amount,
  CAST(proteinAmount AS DOUBLE) AS protein_amount,
  CAST(saltAmount AS DOUBLE) AS salt_amount

FROM hybrid_test.default.product_nutritiens_bronze
WHERE energyAmount IS NOT NULL
  AND caloriesAmount IS NOT NULL
  AND fatAmount IS NOT NULL
  AND saturatedFatAmount IS NOT NULL
  AND carbohydratesAmount IS NOT NULL
  AND sugarsAmount IS NOT NULL
  AND proteinAmount IS NOT NULL
  AND saltAmount IS NOT NULL;

In [0]:
%sql
SELECT *
FROM product_nutritiens_silver
LIMIT 10;

# GOLD

- Adding the first part of the link, for website/image links
- Changing brand values to 'Ikke spesifisert' hvis verdien er null
- Changing description values to 'Ingen beskrivelse' hvis verdien er null
- Multiplying unit values with 10

In [0]:
%sql
CREATE OR REPLACE TABLE product_nutritiens_gold AS
SELECT
  chain,
  title,

  CASE
    WHEN brand IS NULL THEN 'Ikke spesifisert'
    ELSE brand
  END AS brand,

  CASE
    WHEN description IS NULL THEN 'Ingen beskrivelse'
    ELSE description
  END AS description,

  CASE
    WHEN LOWER(chain) = 'spar' THEN concat('https://spar.no', website_url)
    ELSE concat('https://meny.no', website_url)
  END AS website_url,

  concat(
    'https://bilder.ngdata.no/',
    regexp_replace(image_path, '^/+', ''),
    '/large.jpg'
  ) AS image_url,

  price_per_unit,
  compare_price_per_unit,
  compare_unit,

  subtitle,
  ean,

  energy_amount,
  calories_amount,
  fat_amount,
  saturated_fat_amount,
  carbohydrates_amount,
  sugars_amount,
  protein_amount,
  salt_amount,

  energy_amount * 10 AS energy_per_package,
  calories_amount * 10 AS calories_per_package,
  fat_amount * 10 AS fat_per_package,
  saturated_fat_amount * 10 AS saturated_fat_per_package,
  carbohydrates_amount * 10 AS carbohydrates_per_package,
  sugars_amount * 10 AS sugars_per_package,
  protein_amount * 10 AS protein_per_package,
  salt_amount * 10 AS salt_per_package,

  ROUND((energy_amount * 10) / compare_price_per_unit, 2) AS energy_per_nok,
  ROUND((calories_amount * 10) / compare_price_per_unit, 2) AS calories_per_nok,
  ROUND((fat_amount * 10) / compare_price_per_unit, 2) AS fat_per_nok,
  ROUND((saturated_fat_amount * 10) / compare_price_per_unit, 2) AS saturated_fat_per_nok,
  ROUND((carbohydrates_amount * 10) / compare_price_per_unit, 2) AS carbohydrates_per_nok,
  ROUND((sugars_amount * 10) / compare_price_per_unit, 2) AS sugars_per_nok,
  ROUND((protein_amount * 10) / compare_price_per_unit, 2) AS protein_per_nok,
  ROUND((salt_amount * 10) / compare_price_per_unit, 2) AS salt_per_nok

FROM hybrid_test.default.product_nutritiens_silver;

In [0]:
%sql
SELECT *
FROM product_nutritiens_gold
LIMIT 3;

# RESULT

## Products

In [0]:
%sql
SELECT *
FROM products
LIMIT 3;

## Bronze

In [0]:
%sql
SELECT *
FROM product_nutritiens_bronze
LIMIT 3;

## Silver

In [0]:
%sql
SELECT *
FROM product_nutritiens_silver
LIMIT 3;

## Gold

In [0]:
%sql
SELECT *
FROM product_nutritiens_gold
LIMIT 3;